In [0]:
print(f"Spark version: {spark.version}")
print(f"Cluster ready: True")

In [0]:
# Download the file to DBFS
dbutils.fs.cp(
    "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet",
    "dbfs:/tmp/yellow_tripdata_2023-01.parquet"
)

# Load January 2023 yellow taxi data
df_raw = spark.read.parquet(
    "dbfs:/tmp/yellow_tripdata_2023-01.parquet"
)

print(f"Rows:    {df_raw.count():,}")
print(f"Columns: {len(df_raw.columns)}")
df_raw.printSchema()

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, to_date

# Add Bronze metadata columns
df_bronze = df_raw.withColumn("_ingestion_ts", current_timestamp()) \
                  .withColumn("_source_file", input_file_name()) \
                  .withColumn("trip_date", to_date("tpep_pickup_datetime"))

# Write to Bronze Delta table - partition by DATE not timestamp
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("trip_date") \
    .save("dbfs:/portfolio/bronze/yellow_taxi")

print("Bronze write complete")

In [0]:
# Verify Bronze table
df_check = spark.read.format("delta").load("dbfs:/portfolio/bronze/yellow_taxi")
print(f"Bronze rows: {df_check.count():,}")
print(f"Columns:     {len(df_check.columns)}")
print(f"Partitions:  {df_check.select('trip_date').distinct().count()}")

In [0]:
%sql
-- This is your first portfolio screenshot moment
DESCRIBE HISTORY delta.`dbfs:/portfolio/bronze/yellow_taxi`

In [0]:
from pyspark.sql.functions import col, when, lit, unix_timestamp
from pyspark.sql.types import IntegerType

# Read from Bronze
df_bronze = spark.read.format("delta").load("dbfs:/portfolio/bronze/yellow_taxi")

# Silver transformations
df_silver = df_bronze \
    .filter(col("passenger_count") > 0) \
    .filter(col("trip_distance") > 0) \
    .filter(col("fare_amount") > 0) \
    .filter(col("tpep_pickup_datetime").isNotNull()) \
    .filter(col("tpep_dropoff_datetime").isNotNull()) \
    .withColumn("passenger_count", col("passenger_count").cast(IntegerType())) \
    .withColumn("payment_type", col("payment_type").cast(IntegerType())) \
    .withColumn("trip_duration_mins",
    (unix_timestamp("tpep_dropoff_datetime") - 
     unix_timestamp("tpep_pickup_datetime")) / 60
    ) \
    .filter(col("trip_duration_mins") > 0) \
    .filter(col("trip_duration_mins") < 360) \
    .drop("_source_file")

print(f"Silver rows after cleaning: {df_silver.count():,}")
print(f"Rows removed: {3_066_766 - df_silver.count():,}")

In [0]:
# Write Silver Delta table
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("trip_date") \
    .save("dbfs:/portfolio/silver/yellow_taxi")

print("Silver write complete")

In [0]:
%sql
DESCRIBE HISTORY delta.`dbfs:/portfolio/silver/yellow_taxi`

In [0]:
from pyspark.sql.functions import hour, dayofweek, count, avg, sum, round

# Read from Silver
df_silver = spark.read.format("delta").load("dbfs:/portfolio/silver/yellow_taxi")

# Gold table 1 - Hourly revenue by pickup zone
df_hourly_revenue = df_silver \
    .withColumn("pickup_hour", hour("tpep_pickup_datetime")) \
    .groupBy("trip_date", "pickup_hour", "PULocationID") \
    .agg(
        count("*").alias("total_trips"),
        round(sum("fare_amount"), 2).alias("total_fare"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        round(avg("trip_distance"), 2).alias("avg_distance")
    )

# Gold table 2 - Avg trip duration by day of week
df_day_of_week = df_silver \
    .withColumn("day_of_week", dayofweek("tpep_pickup_datetime")) \
    .groupBy("day_of_week") \
    .agg(
        count("*").alias("total_trips"),
        round(avg("trip_duration_mins"), 2).alias("avg_duration_mins"),
        round(avg("fare_amount"), 2).alias("avg_fare")
    ) \
    .orderBy("day_of_week")

# Gold table 3 - Top 10 pickup locations
df_top_locations = df_silver \
    .groupBy("PULocationID") \
    .agg(
        count("*").alias("total_trips"),
        round(avg("fare_amount"), 2).alias("avg_fare"),
        round(sum("total_amount"), 2).alias("total_revenue")
    ) \
    .orderBy(col("total_trips").desc()) \
    .limit(10)

print("Gold dataframes ready")

In [0]:
# Write Gold tables
df_hourly_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("trip_date") \
    .save("dbfs:/portfolio/gold/hourly_revenue")

df_day_of_week.write \
    .format("delta") \
    .mode("overwrite") \
    .save("dbfs:/portfolio/gold/day_of_week")

df_top_locations.write \
    .format("delta") \
    .mode("overwrite") \
    .save("dbfs:/portfolio/gold/top_locations")

print("Gold write complete")

In [0]:
%sql
-- Run before query to capture baseline
DESCRIBE DETAIL delta.`dbfs:/portfolio/gold/hourly_revenue`

In [0]:
%sql
-- Optimize and Z-Order by most common query columns
OPTIMIZE delta.`dbfs:/portfolio/gold/hourly_revenue`
ZORDER BY (PULocationID, pickup_hour)